# Gravitino IRC — Interactive Exploration

This notebook lets you explore the Gravitino Iceberg REST Catalog interactively.
All three engines (Spark, Trino, DuckDB) and PyIceberg point to the same IRC endpoint.

**Prerequisites:** Run at least one engine demo first:
```
make spark-demo      # creates demo_spark.sales
make pyiceberg-demo  # creates demo_pyiceberg.customers
make trino-demo      # creates demo_trino.orders
```

In [ ]:
import os
import pyarrow as pa
import pandas as pd
import duckdb
from pyiceberg.catalog import load_catalog

# ── Connection config ─────────────────────────────────────────────────────────
IRC_URI    = os.environ.get('IRC_URI', 'http://gravitino-irc:9001/iceberg')
S3_ENDPOINT = 'http://minio:9000'
S3_KEY      = os.environ.get('AWS_ACCESS_KEY_ID', 'minioadmin')
S3_SECRET   = os.environ.get('AWS_SECRET_ACCESS_KEY', 'minioadmin123')

print(f'IRC endpoint: {IRC_URI}')
print(f'S3 endpoint:  {S3_ENDPOINT}')

In [ ]:
# ── Connect to Gravitino IRC ──────────────────────────────────────────────────
catalog = load_catalog(
    'gravitino_irc',
    **{
        'type': 'rest',
        'uri': IRC_URI,
        's3.endpoint': S3_ENDPOINT,
        's3.access-key-id': S3_KEY,
        's3.secret-access-key': S3_SECRET,
        's3.path-style-access': 'true',
        's3.region': 'us-east-1',
    }
)
print('Connected to Gravitino IRC:', type(catalog).__name__)

In [ ]:
# ── List all namespaces and tables ────────────────────────────────────────────
print('=== Catalog Contents ===')
for ns in catalog.list_namespaces():
    ns_name = '.'.join(ns)
    tables = catalog.list_tables(ns)
    print(f'\n📁 {ns_name}')
    for tbl in tables:
        print(f'   📄 {"." .join(tbl)}')

In [ ]:
# ── Load a table and inspect its schema ──────────────────────────────────────
# Change namespace/table to explore others
NAMESPACE = 'demo_pyiceberg'
TABLE     = 'customers'

table = catalog.load_table((NAMESPACE, TABLE))
print(f'Table: {NAMESPACE}.{TABLE}')
print(f'Location: {table.location()}')
print(f'Format version: {table.format_version}')
print(f'\nSchema:')
for field in table.schema().fields:
    print(f'  {field.field_id:>3}  {field.name:<25} {field.field_type}')

In [ ]:
# ── Read data into Pandas ─────────────────────────────────────────────────────
df = table.scan().to_arrow().to_pandas()
print(f'Rows: {len(df)}')
df

In [ ]:
# ── Predicate pushdown (server-side filtering) ────────────────────────────────
from pyiceberg.expressions import EqualTo, And, GreaterThanOrEqual

# Only US customers with lifetime_value >= 10000
filtered = table.scan(
    row_filter=And(
        EqualTo('country', 'US'),
        GreaterThanOrEqual('lifetime_value', 10000.0)
    )
).to_arrow().to_pandas()

print(f'US high-value customers: {len(filtered)}')
filtered

In [ ]:
# ── Snapshot history (time travel) ────────────────────────────────────────────
print('Snapshot history:')
for snap in table.snapshots():
    print(f'  ID: {snap.snapshot_id}  |  op: {snap.summary.get("operation", "?")}  |  ts: {snap.timestamp_ms}')

In [ ]:
# ── Read at a specific snapshot (time travel) ─────────────────────────────────
snapshots = table.snapshots()
if len(snapshots) > 1:
    first_snapshot_id = snapshots[0].snapshot_id
    old_df = table.scan(snapshot_id=first_snapshot_id).to_arrow().to_pandas()
    print(f'Rows at first snapshot ({first_snapshot_id}): {len(old_df)}')
    old_df
else:
    print('Only one snapshot exists. Run the demo script again to generate more.')

In [ ]:
# ── DuckDB analytics on IRC data ──────────────────────────────────────────────
conn = duckdb.connect()
conn.execute('INSTALL iceberg; LOAD iceberg;')
conn.execute('INSTALL httpfs; LOAD httpfs;')

conn.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE        s3,
        KEY_ID      '{S3_KEY}',
        SECRET      '{S3_SECRET}',
        ENDPOINT    '{S3_ENDPOINT.replace("http://", "")}',
        URL_STYLE   'path',
        USE_SSL     false,
        REGION      'us-east-1'
    );
""")

# Use PyIceberg to get the metadata location, then pass to DuckDB
metadata_loc = table.metadata_location
print(f'Metadata location: {metadata_loc}')

result = conn.execute(f"""
    SELECT country, tier, COUNT(*) as n, ROUND(AVG(lifetime_value), 2) as avg_ltv
    FROM iceberg_scan('{metadata_loc}', allow_moved_paths=true)
    GROUP BY country, tier
    ORDER BY avg_ltv DESC
""").fetchdf()
print('\nDuckDB aggregation over Iceberg data:')
result

In [ ]:
# ── Write new data from notebook ──────────────────────────────────────────────
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
new_rows = pa.table({
    'customer_id':     pa.array(['C009', 'C010'], type=pa.string()),
    'name':            pa.array(['Iris Park', 'Jack Tan'], type=pa.string()),
    'email':           pa.array(['iris@nb.com', 'jack@nb.com'], type=pa.string()),
    'country':         pa.array(['US', 'CA'], type=pa.string()),
    'tier':            pa.array(['GOLD', 'SILVER'], type=pa.string()),
    'total_orders':    pa.array([20, 8], type=pa.int32()),
    'lifetime_value':  pa.array([5500.0, 1200.0], type=pa.float64()),
    'active':          pa.array([True, True], type=pa.bool_()),
    'created_at':      pa.array([now, now], type=pa.timestamp('us', tz='UTC')),
    'updated_at':      pa.array([now, now], type=pa.timestamp('us', tz='UTC')),
    'referral_code':   pa.array(['REF-NB1', 'REF-NB2'], type=pa.string()),
})

table.append(new_rows)

final_df = table.scan().to_arrow().to_pandas()
print(f'Total rows after notebook append: {len(final_df)}')
final_df.tail(3)